# Allele Frequency vs. Fitness Score

Generates a 2D binned heatmap of log10(allele frequency) vs. SGE fitness score, colored by variant count per bin.

**Required data:** `scores` sheet from the pipeline output Excel file (`*_data.xlsx`). The sheet must contain `gnomad_af` and/or `regeneron_af` columns — these are only present when the pipeline was run with `--excel` and gnomAD/Regeneron input files.

**Required packages:** `sge-utils` (SimpleSGEViz) installed in the active environment.

**Saving:** PNG/SVG require `vl-convert-python` (`pip install vl-convert-python`). Change the extension to `.html` for interactive output (no extra package needed).

In [ ]:
import numpy as np
import pandas as pd
from sgeviz.figures import maf_score

In [ ]:
# --- Configuration ---
excel_path = "GENE_data.xlsx"         # path to pipeline output Excel file
gene = "GENE"                         # gene name for plot title
output_path = f"{gene}_maf_vs_score.png"  # change to .html for interactive output, or .svg

# --- Plot customization (optional) ---
width = 300     # heatmap width in pixels
height = 250    # heatmap height in pixels

In [ ]:
# --- Load data ---
scores_df = pd.read_excel(excel_path, sheet_name="scores")

# Identify available AF columns
af_col_map = {}
if "gnomad_af" in scores_df.columns:
    af_col_map["gnomad_af"] = "gnomAD"
if "regeneron_af" in scores_df.columns:
    af_col_map["regeneron_af"] = "Regeneron"

if not af_col_map:
    raise ValueError(
        "No allele frequency columns (gnomad_af, regeneron_af) found in scores sheet. "
        "Re-run the pipeline with --excel and gnomAD/Regeneron input files to generate this data."
    )

# Reconstruct long-form maf_df from wide AF columns (SNVs only)
snv_df = scores_df[scores_df["var_type"] == "snv"].copy()
dfs = []
for col, dataset in af_col_map.items():
    sub = snv_df[["score", col]].dropna().copy()
    sub = sub[sub[col] > 0]
    sub["dataset"] = dataset
    sub = sub.rename(columns={col: "Allele Frequency"})
    dfs.append(sub)

maf_df = pd.concat(dfs, ignore_index=True)
maf_df["log_AF"] = np.log10(maf_df["Allele Frequency"])
print(f"Loaded {len(maf_df)} variants with allele frequency data")

In [ ]:
# --- Generate plot ---
chart = maf_score.make_plot(maf_df, gene=gene, width=width, height=height)
chart

In [ ]:
# --- Save ---
chart.save(output_path)
print(f"Saved: {output_path}")